=============================================================================
98110 Mobility Demand Analysis 
=============================================================================
To what extent does traffic demand to/from commercial areas exist outside
transit operating hours?
=============================================================================

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import seaborn as sns

In [ ]:
# Set up formatting
pd.options.mode.chained_assignment = None
sns.set_style("whitegrid")
comma_fmt = FuncFormatter(lambda x, p: format(int(x), ','))

=============================================================================
CONFIGURATION - EDIT THESE 
=============================================================================

In [ ]:
# File paths
INPUT_FILE = r'C:\Users\rebeccasc\Documents\Scripts\BI_commerce_TDM\2013907_BI_Corridor_Volume_2025\2013907_BI_Corridor_Volume_2024_mf_all.csv'
OUTPUT_DIR = r'C:\Users\rebeccasc\Documents\Scripts\BI_commerce_TDM\output'

In [ ]:
# Create output subdirectories
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'heatmaps'), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'summary_tables'), exist_ok=True)

=============================================================================
TRANSIT OPERATING HOURS 
=============================================================================

In [ ]:
TRANSIT_CONFIG = {
    'weekday': {
        'start_hour': 4.5,  # 4:30 AM
        'end_hour': 19.25,  # 7:15 PM
        'name': 'Weekday'
    },
    'saturday': {
        'start_hour': 9,    # 9:00 AM
        'end_hour': 18,     # 6:00 PM
        'name': 'Saturday'
    },
    'sunday': {
        'start_hour': 9,    # 9:00 AM
        'end_hour': 16,     # 4:00 PM
        'name': 'Sunday'
    }
}

In [ ]:
# Day types to analyze (excluding the aggregate "All Days")
DAY_TYPES_TO_ANALYZE = [
    '1: Monday (M-M)',
    '2: Tuesday (Tu-Tu)',
    '3: Wednesday (W-W)',
    '4: Thursday (Th-Th)',
    '5: Friday (F-F)',
    '6: Saturday (Sa-Sa)',
    '7: Sunday (Su-Su)'
]

In [ ]:
# Commercial/middle filter zones  
COMMERCIAL_CORRIDORS = [
    'HIGH_SCHOOL_RD',
    'SPORTSMAN_CLUB',
    'SR305_MID',
    'SR305_N',
    'SR305_S',
    'WINSLOW_WAY'
]

In [ ]:
# Origin and destination zones
ORIGIN_ZONE = 'kitsap_broadly2'
DESTINATION_ZONE = 'kitsap_broadly4'

In [ ]:
# Volume column
VOLUME_COL = 'Average Daily O-M-D Traffic (StL Volume)'

In [ ]:
# Time periods for summary tables (approx time bands) 
TIME_PERIODS = {
    'All Day': {
        'hours': list(range(0, 24)),
        'display': 'All Day (24hr)'
    },
    'Early AM': {
        'hours': [4, 5, 6],
        'display': 'Early Morning (4am-7am)'
    },
    'Morning Peak': {
        'hours': [7, 8, 9],
        'display': 'Morning Peak (7am-10am)'
    },
    'Midday': {
        'hours': [10, 11, 12, 13, 14],
        'display': 'Midday (10am-3pm)'
    },
    'Evening Peak': {
        'hours': [15, 16, 17, 18],
        'display': 'Evening Peak (3pm-7pm)'
    },
    'Late Evening': {
        'hours': [19, 20, 21],
        'display': 'Late Evening (7pm-10pm)'
    },
    'Late Night': {
        'hours': [22, 23, 0, 1, 2, 3],
        'display': 'Late Night (10pm-4am)'
    }
}

=============================================================================
HELPER FUNCTIONS
=============================================================================

In [ ]:
def extract_hour(period_str):
    """Extract hour number from period string e.g., '08: 7am-8am'"""
    try:
        return int(period_str[:2])
    except:
        return None

In [ ]:
def get_time_period_name(hour):
    """Return custom time period per hour """
    if hour is None:
        return 'Daily Total'
    for period_name, period_info in TIME_PERIODS.items():
        if hour in period_info['hours']:
            return period_name
    return 'Other'

In [ ]:
def get_transit_hours(day_type):
    """Get transit hours based on day type"""
    day_type_lower = day_type.lower()
    
    if any(x in day_type_lower for x in ['monday', 'tuesday', 'wednesday', 'thursday', 'friday']):
        return TRANSIT_CONFIG['weekday']
    elif 'saturday' in day_type_lower:
        return TRANSIT_CONFIG['saturday']
    elif 'sunday' in day_type_lower:
        return TRANSIT_CONFIG['sunday']
    else:
        return TRANSIT_CONFIG['weekday']  # Default

In [ ]:
def get_transit_status(hour, day_type):
    """Return transit status for a given hour and day type"""
    if hour is None:
        return 'Daily Total'
    
    transit = get_transit_hours(day_type)
    
    if transit is None:
        return 'No Service'
    elif hour < transit['start_hour']:
        return 'Before Service'
    elif hour < transit['end_hour']:
        return 'During Service'
    else:
        return 'After Service'

=============================================================================
PREP DATA
=============================================================================

In [ ]:
print("=" * 80)
print("98110 MOBILITY DEMAND ANALYSIS")
print("=" * 80)

In [ ]:
print(f"\nReading data from: {INPUT_FILE}")
df = pd.read_csv(INPUT_FILE)

In [ ]:
# Add hour column (will be None for 'All Day' records)
df['Hour'] = df['Day Part'].apply(extract_hour)

In [ ]:
# Add time period column
df['Time_Period'] = df['Hour'].apply(get_time_period_name)

In [ ]:
# Separate hourly data (excluding 'All Day' records)
df_hourly = df[df['Day Part'] != '00: All Day (12am-12am)'].copy()
print(f"\nHourly records (for heatmaps & peak analysis): {len(df_hourly):,}")

In [ ]:
# Separate daily totals (only 'All Day' records)
df_daily = df[df['Day Part'] == '00: All Day (12am-12am)'].copy()
print(f"Daily total records (for summary tables): {len(df_daily):,}")

In [ ]:
# Filter out the aggregate "All Days" from both datasets
df_hourly_individual = df_hourly[~df_hourly['Day Type'].str.contains('All Days', na=False)].copy()
df_daily_individual = df_daily[~df_daily['Day Type'].str.contains('All Days', na=False)].copy()

In [ ]:
print(f"\nData Overview:")
print(f"  Hourly data - individual days: {len(df_hourly_individual):,} records")
print(f"  Daily totals - individual days: {len(df_daily_individual):,} records")
print(f"  Day types: {df_hourly_individual['Day Type'].unique().tolist()}")
print(f"  Hours range: {df_hourly_individual['Hour'].min()} to {df_hourly_individual['Hour'].max()}")

=============================================================================
FUNCTION TO CREATE SUMMARY TABLES (using HOURLY AVERAGE data)
=============================================================================

In [ ]:
def create_summary_table(data_df, period_name, period_info):
    """
    Create a summary table of AVERAGE volumes by Day Type and Corridor for a specific time period
    Since the data contains AVERAGES, we average the hourly averages within each time period
    """
    # Filter for this time period's hours
    if period_name != 'All Day':
        period_data = data_df[data_df['Hour'].isin(period_info['hours'])]
    else:
        period_data = data_df  # All Day uses all hours
    
    if len(period_data) == 0:
        print(f"  No data for {period_info['display']}")
        return None
    
    # For averages, we need to take the mean of the hourly values in this period
    summary = period_data.pivot_table(
        values=VOLUME_COL,
        index='Middle Filter Zone Name',
        columns='Day Type',
        aggfunc='mean',  # Use mean instead of sum for averages!
        fill_value=0
    )
    
    # Reorder rows and columns
    summary = summary.reindex(COMMERCIAL_CORRIDORS)
    summary = summary[[col for col in DAY_TYPES_TO_ANALYZE if col in summary.columns]]
    
    # Add average row and column (averages of averages - use with caution)
    summary.loc['AVERAGE'] = summary.mean()
    summary['AVERAGE'] = summary.mean(axis=1)
    
    return summary

=============================================================================
CREATE AND EXPORT SUMMARY TABLES
=============================================================================

In [ ]:
print("\n" + "=" * 80)
print("CREATING SUMMARY TABLES")
print("=" * 80)

In [ ]:
summary_tables = {}

In [ ]:
# Create summary tables for each time period using HOURLY data
for period_name, period_info in TIME_PERIODS.items():
    print(f"\n{period_info['display']}:")
    
    summary = create_summary_table(df_hourly_individual, period_name, period_info)
    
    if summary is not None:
        summary_tables[period_name] = summary
        
        # Save to CSV
        csv_path = os.path.join(OUTPUT_DIR, 'summary_tables', f'summary_{period_name}.csv')
        summary.to_csv(csv_path)
        
        # Display preview
        print(summary.round(0).astype(int))
        print(f"  Saved to: {csv_path}")

=============================================================================
HEATMAP 1: HOURLY PATTERNS BY DAY TYPE FOR EACH CORRIDOR
=============================================================================

In [ ]:
print("\n" + "=" * 80)
print("CREATING CORRIDOR HEATMAPS")
print("=" * 80)

In [ ]:
for corridor in COMMERCIAL_CORRIDORS:
    print(f"\nCreating heatmap for {corridor}...")
    
    # Filter hourly data for this corridor
    corridor_data = df_hourly_individual[df_hourly_individual['Middle Filter Zone Name'] == corridor]
    
    if len(corridor_data) == 0:
        print(f"  No data for {corridor}")
        continue
    
    # Create pivot table: hours vs day types
    pivot_data = corridor_data.pivot_table(
        values=VOLUME_COL,
        index='Hour',
        columns='Day Type',
        aggfunc='mean',  # Use mean since these are averages
        fill_value=0
    )
    
    # Reorder columns to match day order
    pivot_data = pivot_data[[col for col in DAY_TYPES_TO_ANALYZE if col in pivot_data.columns]]
    
    # Find the maximum value in this specific dataset for color scaling
    max_value = pivot_data.max().max()
    
    print(f"  Value range: 0 to {max_value:.0f}")
    
    # Create heatmap with dynamic color scaling
    plt.figure(figsize=(14, 8))
    
    # Plot heatmap
    ax = sns.heatmap(pivot_data, 
                     annot=True, 
                     fmt='.0f',
                     cmap='YlOrRd',
                     linewidths=0.5,
                     vmin=0,
                     vmax=max_value,
                     cbar_kws={'label': 'Average Volume', 'format': comma_fmt})
    
    # Add transit hour indicators
    for i, day_type in enumerate(pivot_data.columns):
        transit = get_transit_hours(day_type)
        if transit:
            start_row = int(transit['start_hour'])
            end_row = int(transit['end_hour'])
            ax.add_patch(plt.Rectangle((i, start_row), 1, end_row-start_row,
                                       fill=False, edgecolor='green', linewidth=2, linestyle='--'))
    
    plt.title(f'Hourly Traffic Volumes - {corridor}\n(Transit hours outlined in green dashed lines)', fontsize=14)
    plt.xlabel('Day Type')
    plt.ylabel('Hour of Day')
    plt.tight_layout()
    
    # Save
    filename = f'Heatmap_{corridor}.pdf'
    filepath = os.path.join(OUTPUT_DIR, 'heatmaps', filename)
    plt.savefig(filepath, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"  Saved: {filename}")

=============================================================================
HEATMAP 2: SUMMARY BY CORRIDOR AND DAY TYPE
=============================================================================

In [ ]:
print("\n" + "=" * 80)
print("CREATING SUMMARY HEATMAP")
print("=" * 80)

In [ ]:
# Create summary pivot using the 'All Day' summary table
if 'All Day' in summary_tables:
    summary_pivot = summary_tables['All Day'].drop('AVERAGE', axis=0).drop('AVERAGE', axis=1)
    
    max_value = summary_pivot.max().max()
    
    plt.figure(figsize=(12, 6))
    sns.heatmap(summary_pivot, 
                annot=True, 
                fmt='.0f',
                cmap='YlOrRd',
                linewidths=0.5,
                vmin=0,
                vmax=max_value,
                cbar_kws={'label': 'Average Daily Volume', 'format': comma_fmt})
    
    plt.title('Average Daily Traffic Volume by Corridor and Day Type', fontsize=14)
    plt.xlabel('Day Type')
    plt.ylabel('Corridor')
    plt.tight_layout()
    
    filename = 'Heatmap_Summary.pdf'
    filepath = os.path.join(OUTPUT_DIR, 'heatmaps', filename)
    plt.savefig(filepath, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"Saved: {filename} (color scale: 0 to {max_value:.0f})")

=============================================================================
PEAK HOUR ANALYSIS
=============================================================================

In [ ]:
print("\n" + "=" * 80)
print("PEAK HOUR ANALYSIS")
print("=" * 80)

In [ ]:
# Add transit status column for hourly analysis
df_hourly_individual['Transit_Status'] = df_hourly_individual.apply(
    lambda row: get_transit_status(row['Hour'], row['Day Type']), 
    axis=1
)

In [ ]:
peak_results = []

In [ ]:
for corridor in COMMERCIAL_CORRIDORS:
    for day_type in DAY_TYPES_TO_ANALYZE:
        day_data = df_hourly_individual[
            (df_hourly_individual['Middle Filter Zone Name'] == corridor) & 
            (df_hourly_individual['Day Type'] == day_type)
        ]
        
        if len(day_data) == 0:
            continue
        
        # Find peak hour
        max_idx = day_data[VOLUME_COL].idxmax()
        peak_hour = day_data.loc[max_idx, 'Hour']
        peak_volume = day_data.loc[max_idx, VOLUME_COL]
        
        # Get transit status for peak hour
        transit_status = get_transit_status(peak_hour, day_type)
        
        peak_results.append({
            'Corridor': corridor,
            'Day_Type': day_type,
            'Peak_Hour': f"{int(peak_hour):02d}:00-{int(peak_hour)+1:02d}:00",
            'Peak_Volume': peak_volume,
            'Transit_Status': transit_status
        })

In [ ]:
peak_df = pd.DataFrame(peak_results)
print(f"\nTotal peak hours identified: {len(peak_df)}")
print("\nSample of Peak Hours Analysis (first 10 rows):")
print(peak_df.head(10).to_string(index=False))

=============================================================================
TRANSIT COVERAGE ANALYSIS
=============================================================================

In [ ]:
print("\n" + "=" * 80)
print("TRANSIT COVERAGE ANALYSIS")
print("=" * 80)

In [ ]:
# Overall statistics using hourly data
total_vol = df_hourly_individual[VOLUME_COL].sum()
status_summary = df_hourly_individual.groupby('Transit_Status')[VOLUME_COL].sum()

In [ ]:
print("\nOverall Traffic by Transit Status:")
for status in ['Before Service', 'During Service', 'After Service']:
    if status in status_summary.index:
        vol = status_summary[status]
        print(f"  {status}: {vol:,.0f} ({vol/total_vol*100:.1f}%)")

In [ ]:
# By corridor
print("\n" + "-" * 60)
print("DETAILED ANALYSIS BY CORRIDOR")
print("-" * 60)

In [ ]:
corridor_summary = []
for corridor in COMMERCIAL_CORRIDORS:
    corridor_data = df_hourly_individual[df_hourly_individual['Middle Filter Zone Name'] == corridor]
    
    if len(corridor_data) == 0:
        continue
    
    total = corridor_data[VOLUME_COL].sum()
    by_status = corridor_data.groupby('Transit_Status')[VOLUME_COL].sum()
    
    row = {'Corridor': corridor, 'Total_Volume': total}
    for status in ['Before Service', 'During Service', 'After Service']:
        row[status] = by_status.get(status, 0)
        row[f'Pct_{status}'] = (by_status.get(status, 0) / total * 100) if total > 0 else 0
    
    corridor_summary.append(row)

In [ ]:
corridor_df = pd.DataFrame(corridor_summary)
print("\nCorridor Analysis (% outside transit hours):")
for _, row in corridor_df.iterrows():
    outside_pct = 100 - row['Pct_During Service']
    print(f"  {row['Corridor']}: {outside_pct:.1f}% outside transit hours")
    print(f"    - Before Service: {row['Pct_Before Service']:.1f}%")
    print(f"    - After Service: {row['Pct_After Service']:.1f}%")

=============================================================================
EXPORT ALL RESULTS
=============================================================================

In [ ]:
print("\n" + "=" * 80)
print("EXPORTING RESULTS")
print("=" * 80)

In [ ]:
# Create Excel file with all analyses
with pd.ExcelWriter(os.path.join(OUTPUT_DIR, 'complete_analysis.xlsx')) as writer:
    # Summary tables
    for period_name, summary in summary_tables.items():
        summary.to_excel(writer, sheet_name=f'Summary_{period_name}')
    
    # Peak hours
    peak_df.to_excel(writer, sheet_name='Peak_Hours', index=False)
    
    # Corridor summary
    corridor_df.to_excel(writer, sheet_name='Corridor_Summary', index=False)
    
    # Hourly data pivot
    hourly_pivot = df_hourly_individual.pivot_table(
        values=VOLUME_COL,
        index=['Middle Filter Zone Name', 'Hour'],
        columns='Day Type',
        aggfunc='mean',
        fill_value=0
    )
    hourly_pivot.to_excel(writer, sheet_name='Hourly_Data')

In [ ]:
print(f"\nAll results saved to: {os.path.join(OUTPUT_DIR, 'complete_analysis.xlsx')}")
print(f"Heatmaps saved to: {os.path.join(OUTPUT_DIR, 'heatmaps')}")
print(f"Summary tables saved to: {os.path.join(OUTPUT_DIR, 'summary_tables')}")

=============================================================================
KEY FINDINGS 
=============================================================================

In [ ]:
print("\n" + "=" * 80)
print("KEY FINDINGS ")
print("=" * 80)

In [ ]:
# Calculate key metrics
during_pct = (status_summary.get('During Service', 0) / total_vol * 100)
before_pct = (status_summary.get('Before Service', 0) / total_vol * 100)
after_pct = (status_summary.get('After Service', 0) / total_vol * 100)
outside_total = before_pct + after_pct

In [ ]:
# Find corridors with most outside-transit demand
corridor_df['Outside_Transit_Pct'] = 100 - corridor_df['Pct_During Service']
top_outside = corridor_df.nlargest(3, 'Outside_Transit_Pct')

In [ ]:
# Find peaks outside transit
outside_peaks = peak_df[peak_df['Transit_Status'] != 'During Service'].head(5)

In [ ]:
print(f"""
{'='*60}
98110 MOBILITY DEMAND ANALYSIS RESULTS
{'='*60}

1. TRANSIT COVERAGE SUMMARY:
   - During transit hours: {during_pct:.1f}% of total traffic
   - Outside transit hours: {outside_total:.1f}% of total traffic
     * Before service starts: {before_pct:.1f}%
     * After service ends: {after_pct:.1f}%

2. CORRIDORS WITH HIGHEST DEMAND OUTSIDE TRANSIT HOURS:
""")

In [ ]:
for _, row in top_outside.iterrows():
    print(f"   • {row['Corridor']}: {row['Outside_Transit_Pct']:.1f}% outside transit hours")

In [ ]:
print(f"""
3. PEAK HOURS OUTSIDE CURRENT TRANSIT SERVICE:
""")

In [ ]:
if len(outside_peaks) > 0:
    for _, row in outside_peaks.iterrows():
        print(f"   • {row['Corridor']} on {row['Day_Type']}: {row['Peak_Hour']} ({row['Peak_Volume']:,.0f} vehicles)")
else:
    print("   No peak hours found outside transit service")

In [ ]:
print(f"""
4. RECOMMENDATIONS:

   Based on the analysis of {len(df_hourly_individual):,} hourly records across {len(COMMERCIAL_CORRIDORS)} corridors:
""")

In [ ]:
if before_pct > 10:
    print(f"   • EARLY MORNING SERVICE: Consider extending service earlier than {int(TRANSIT_CONFIG['weekday']['start_hour']):02d}:00")
    print(f"     ({before_pct:.1f}% of traffic occurs before service starts)")

In [ ]:
if after_pct > 10:
    print(f"   • EVENING SERVICE: Consider extending service later than {int(TRANSIT_CONFIG['weekday']['end_hour']):02d}:00")
    print(f"     ({after_pct:.1f}% of traffic occurs after service ends)")

In [ ]:
if outside_total > 30:
    print(f"\n   • GAP: {outside_total:.1f}% of all traffic occurs outside transit hours")
    print(f"     This suggests strong potential demand for expanded service hours.")

In [ ]:
print(f"""
5. NEXT STEPS:
   • Review corridor heatmaps in the 'heatmaps' folder
   • Check complete_analysis.xlsx for details
   • Validate peak hours against ferry times
""")

In [ ]:
print("\n" + "=" * 80)
print("IT'S ALL COMPLETE")
print("=" * 80)